# Forecast por segmento financeiro: ARIMA, SARIMA e XGBoost com validação temporal

In [ ]:
# Preparação automática apenas ao executar no Google Colab.
import os as _os
import csv as _csv
import sys as _sys
import subprocess as _subprocess
from pathlib import Path as _Path

try:
    import google.colab
    _executando_no_colab = True
except ImportError:
    _executando_no_colab = False

if _executando_no_colab:
    _pacotes = [
        "kagglehub>=1.0.2", "numpy>=1.26,<3", "pandas>=2.1,<3",
        "matplotlib>=3.8,<4", "scikit-learn>=1.4,<2",
        "statsmodels>=0.14,<1", "xgboost>=2,<4", "shap>=0.45,<1",
    ]
    _subprocess.run([_sys.executable, "-m", "pip", "install", "-q", *_pacotes], check=True)
    import kagglehub

    _caminho = _Path(kagglehub.dataset_download("belbino/indian-banking-transactions-20192024", path="indian_banking_transactions.csv"))
    if _caminho.is_dir():
        _caminho = _caminho / "indian_banking_transactions.csv"
    assert _caminho.is_file(), f"Dataset não encontrado: {_caminho}"
    with _caminho.open("r", encoding="utf-8-sig", newline="") as _arquivo:
        _campos = set(next(_csv.reader(_arquivo)))
    _esperados = set(["transaction_id","customer_id","transaction_date","account_type","merchant_category","transaction_amount","transaction_status"])
    assert _esperados <= _campos, f"Colunas ausentes: {sorted(_esperados - _campos)}"
    _os.environ["FINANCE_BI_BANKING_CSV"] = str(_caminho)
    _os.environ.setdefault("FINANCE_BI_FAST_TEST", "1")
    _os.environ.setdefault("FINANCE_BI_OUTPUT_DIR", "/content/finance_bi_outputs")
    print("Dados Kaggle:", _caminho)
    print("Arquivo validado (cabeçalho):", len(_campos), "colunas")
    print("Modo rápido:", _os.environ["FINANCE_BI_FAST_TEST"] == "1")


In [ ]:
from pathlib import Path
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBRegressor
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

FAST = os.environ.get("FINANCE_BI_FAST_TEST") == "1"
CSV = Path(os.environ.get("FINANCE_BI_BANKING_CSV", "data/raw/indian_banking_transactions.csv"))
OUT = Path(os.environ.get("FINANCE_BI_OUTPUT_DIR", "outputs")) / "02_forecast_segmentos"
OUT.mkdir(parents=True, exist_ok=True)
assert CSV.is_file(), f"CSV não encontrado: {CSV}"
cols = ["transaction_id", "customer_id", "transaction_date", "account_type", "merchant_category", "transaction_amount", "transaction_status"]
df = pd.read_csv(CSV, usecols=cols, parse_dates=["transaction_date"], low_memory=False).drop_duplicates("transaction_id")
df["transaction_amount"] = pd.to_numeric(df["transaction_amount"], errors="coerce")
df = df.dropna(subset=["customer_id", "transaction_date", "transaction_amount"])
df = df.loc[(df["transaction_amount"]>=0)&df["transaction_status"].eq("Success")].copy()
assert not df.empty
# Mês incompleto no início/fim da série não equivale a queda transacional.
_inicio_bruto = df["transaction_date"].min().normalize()
_fim_bruto = df["transaction_date"].max().normalize()
_inicio_util = _inicio_bruto.to_period("M").to_timestamp()
_fim_util = _fim_bruto.to_period("M").to_timestamp()
if _inicio_bruto.day != 1:
    _inicio_util = _inicio_util + pd.offsets.MonthBegin(1)
if _fim_bruto != _fim_bruto + pd.offsets.MonthEnd(0):
    _fim_util = _fim_util - pd.offsets.MonthBegin(1)
_linhas_incompletas = len(df)
df = df.loc[df["transaction_date"].between(_inicio_util, _fim_util + pd.offsets.MonthEnd(0))].copy()
print("Períodos de borda incompletos: excluídas", _linhas_incompletas - len(df),
      "transações | janela mensal:", _inicio_util.date(), "a", _fim_util.date())
assert not df.empty, "Sem períodos mensais completos."
df["mes"] = df["transaction_date"].dt.to_period("M").dt.to_timestamp()
meses = pd.date_range(df["mes"].min(), df["mes"].max(), freq="MS")
assert len(meses) >= 36, "Série insuficiente: requer pelo menos 36 meses."
print("Meses:", len(meses), "| período:", meses[0].date(), "a", meses[-1].date())

In [ ]:
inicio_teste = meses[max(24, len(meses)-18)]
def montar_series(frame, familia, coluna, qtde=2):
    # Seleção do segmento usando somente o período disponível antes do backtesting.
    referencia = frame.loc[frame["mes"] < inicio_teste]
    principais = referencia.groupby(coluna)["transaction_amount"].sum().nlargest(qtde).index
    result = {}
    for nome in principais:
        subset = frame.loc[frame[coluna].eq(nome)]
        nome_serie = f"{familia}:{nome}"
        result[nome_serie] = subset.groupby("mes")["transaction_amount"].sum().reindex(meses, fill_value=0).astype(float)
    return result

series = {"TOTAL": df.groupby("mes")["transaction_amount"].sum().reindex(meses,fill_value=0).astype(float)}
series.update(montar_series(df,"tipo_conta","account_type"))
series.update(montar_series(df,"categoria","merchant_category"))
# Perfis calculados exclusivamente no histórico anterior à janela de testes.
historico = df.loc[df["mes"] < inicio_teste]
perf = historico.groupby("customer_id").agg(n=("transaction_id","count"), volume=("transaction_amount","sum"), ticket=("transaction_amount","mean")).reset_index()
if len(perf)>=6 and perf[["n","volume","ticket"]].drop_duplicates().shape[0]>=2:
    base = np.log1p(perf[["n","volume","ticket"]].clip(lower=0))
    xs = StandardScaler().fit_transform(base)
    perf["perfil"] = KMeans(n_clusters=min(3,len(perf)//3), random_state=42,n_init=10).fit_predict(xs).astype(str)
    com_perfil = df.merge(perf[["customer_id","perfil"]], on="customer_id", how="left")
    com_perfil["perfil"] = com_perfil["perfil"].fillna("novo")
    series.update(montar_series(com_perfil,"perfil","perfil",qtde=2))
if FAST:
    series = dict(list(series.items())[:2])
print("Séries:", list(series))
assert all(s.index.equals(meses) for s in series.values())

In [ ]:
def xgb_pontos(values):
    a = np.asarray(values, dtype=float)
    linhas, y = [], []
    for t in range(12,len(a)):
        dt = meses[0] + pd.DateOffset(months=t)
        linhas.append([a[t-1],a[t-2],a[t-3],a[t-6],a[t-12],np.mean(a[t-3:t]),np.sin(2*np.pi*dt.month/12),np.cos(2*np.pi*dt.month/12)])
        y.append(a[t])
    return np.asarray(linhas,float), np.asarray(y,float)

def prever_xgb(treino, horizonte):
    a = np.asarray(treino,dtype=float)
    assert len(a)>=24
    features, target = xgb_pontos(a)
    model=XGBRegressor(n_estimators=60 if FAST else 140,max_depth=3,learning_rate=.06,subsample=.9,colsample_bytree=.9,random_state=42,n_jobs=2,objective="reg:squarederror")
    model.fit(features,target)
    history=list(a)
    result=[]
    for h in range(horizonte):
        dt = meses[0]+pd.DateOffset(months=len(history))
        feats=np.array([[history[-1],history[-2],history[-3],history[-6],history[-12],np.mean(history[-3:]),np.sin(2*np.pi*dt.month/12),np.cos(2*np.pi*dt.month/12)]],dtype=float)
        val=max(0.,float(model.predict(feats)[0]))
        result.append(val); history.append(val)
    return np.asarray(result)

def prever_arma(treino,horizonte,sazonal=False):
    order = (1,1,1)
    seasonal_order = (0,1,1,12) if sazonal else (0,0,0,0)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = SARIMAX(np.asarray(treino,dtype=float),order=order,seasonal_order=seasonal_order,enforce_stationarity=False,enforce_invertibility=False)
        fit = model.fit(disp=False,maxiter=40 if FAST else 90)
    return np.maximum(0,np.asarray(fit.forecast(horizonte),dtype=float))

def prever(nome,treino,horizonte):
    if nome=="ARIMA":return prever_arma(treino,horizonte,False)
    if nome=="SARIMA":return prever_arma(treino,horizonte,True)
    if nome=="XGBoost":return prever_xgb(treino,horizonte)
    if nome=="Naive sazonal":
        a=np.asarray(treino,dtype=float)
        return np.tile(a[-12:], int(np.ceil(horizonte/12)))[:horizonte]
    raise ValueError(nome)

In [ ]:
registros, erros = [], []
modelos = ["Naive sazonal", "ARIMA", "SARIMA", "XGBoost"]
horizonte=3
origens = [len(meses)-horizonte, len(meses)-2*horizonte, len(meses)-3*horizonte]
if FAST: origens=origens[-2:]
for segmento,y in series.items():
    vals=y.to_numpy(dtype=float)
    for fim in sorted(origens):
        treino, teste=vals[:fim], vals[fim:fim+horizonte]
        for nome in modelos:
            try:
                pred=prever(nome,treino,horizonte)
                assert np.isfinite(pred).all() and (pred>=0).all()
                for h,(real,estimado) in enumerate(zip(teste,pred),1):
                    registros.append({"segmento":segmento,"modelo":nome,"corte_treino":str(meses[fim-1].date()),"horizonte":h,"real_inr":real,"previsto_inr":estimado,"erro_abs":abs(real-estimado)})
            except Exception as exc:
                erros.append({"segmento":segmento,"modelo":nome,"corte_treino":str(meses[fim-1].date()),"erro":str(exc)[:180]})
assert registros, "Nenhum modelo concluiu a validação."
backtest=pd.DataFrame(registros)
resumo=backtest.groupby(["segmento","modelo"],as_index=False).agg(mae=("erro_abs","mean"),volume_real=("real_inr","sum"),erro_total=("erro_abs","sum"),amostras=("real_inr","count"))
resumo["wape"] = np.where(resumo["volume_real"]>0,resumo["erro_total"]/resumo["volume_real"],np.nan)
resumo=resumo.sort_values(["segmento","mae"])
print("Validação por origens cronológicas; previsão recursiva sem acessar os meses de teste:")
display(resumo)
if erros: display(pd.DataFrame(erros))
assert backtest["corte_treino"].nunique()>=2, "Backtesting requer ao menos duas origens temporais."
assert not any(pd.to_datetime(backtest["corte_treino"]).dt.to_period("M") > meses[-1].to_period("M")), "Janela de backtest inválida."


In [ ]:
futuros=[]
for segmento,grupo in resumo.groupby("segmento",sort=False):
    melhor=grupo.iloc[0]["modelo"]
    y=series[segmento]
    try:
        valores=prever(melhor,y.to_numpy(),3)
        for data,valor in zip(pd.date_range(meses[-1]+pd.offsets.MonthBegin(1),periods=3,freq="MS"),valores):
            futuros.append({"segmento":segmento,"modelo_escolhido":melhor,"mes":data,"previsao_volume_inr":valor})
    except Exception as exc:
        print(f"Não foi possível projetar {segmento}: {exc}")
forecast=pd.DataFrame(futuros)
assert not forecast.empty
print("Forecast por segmento:")
display(forecast)
fig,ax=plt.subplots(figsize=(12,5))
y=series["TOTAL"]
ax.plot(y.index,y.values,label="Histórico observado")
proj=forecast.loc[forecast["segmento"].eq("TOTAL")]
if not proj.empty:ax.plot(proj["mes"],proj["previsao_volume_inr"],"o--",label="Previsão selecionada")
ax.set(title="Volume mensal total — histórico e projeção futura",ylabel="Movimentação (INR)")
ax.legend();ax.grid(alpha=.2);plt.tight_layout();plt.show()

In [ ]:
backtest.to_csv(OUT/"backtest_forecast_segmentos.csv",index=False)
resumo.to_csv(OUT/"comparacao_modelos.csv",index=False)
forecast.to_csv(OUT/"forecast_segmentos.csv",index=False)
print("Arquivos:",*[str(p) for p in sorted(OUT.glob('*.csv'))],sep="\n")
print("Dados sintéticos; horizontes, intervalos e eventuais falhas de convergência precisam ser interpretados. Nenhuma série PaySim é misturada com Banking.")